In [29]:
import plotly.graph_objects as go
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import numpy as np

ticker = "MSTR"
start_date = "2025-01-01"
end_date = "2026-06-13"

df = yf.download(ticker, start=start_date, end=end_date)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.to_csv("Stock.csv")

# --- Define Split Ratio ---
train_ratio = 0.8  # 80% for training, 20% for testing
total_days = len(df)
split_idx = int(total_days * train_ratio)

# --- Split the Data Chronologically ---
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

print(f"Total Rows: {total_days}")
print(f"Training Rows (First 80%): {len(train_df)} (From {train_df.index[0].date()} to {train_df.index[-1].date()})")
print(f"Testing Rows (Last 20%): {len(test_df)} (From {test_df.index[0].date()} to {test_df.index[-1].date()})")

# --- Extract Features Separately ---
train_features = train_df[['Open', 'Close']].to_numpy()
test_features = test_df[['Open', 'Close']].to_numpy()

# 1. Fit the scaler on TRAINING data only
train_min = train_features.min(axis=0)
train_max = train_features.max(axis=0)

# 2. Scale the training windows
T = 60
window_data_train = train_features[0:T, :]
scaled_window_train = (window_data_train - train_min) / (train_max - train_min)
inputVar_train = scaled_window_train.T  # Shape: (2, 60)

# 3. Scale the testing windows using TRAINING parameters
window_data_test = test_features[0:T, :]
scaled_window_test = (window_data_test - train_min) / (train_max - train_min)
inputVar_test = scaled_window_test.T  # Shape: (2, 60)

# Input-to-Hidden weights == (3, 2)
WxH = np.array([
    [ 0.5, -0.2],
    [ 0.1,  0.8],
    [-0.4,  0.3]
])  

# Hidden-to-Hidden weights == (3, 3)
WhH = np.array([
    [ 0.9,  0.1, -0.2],
    [ 0.0,  0.8,  0.3],
    [-0.1,  0.2,  0.7]
])  

# Hidden-to-Output weights

WhY = np.array([
    [ 0.4, -0.5,  0.2],
    [-0.3,  0.7,  0.1]
])  

# Biases
hS0 = np.zeros((3, 1))  
hS0_train = np.zeros((3,1))
biasWV = np.zeros((3, 1))
biasWY = np.zeros((2, 1))

In [31]:
# tanh (from scratch)
def tanh(x):
    return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

In [42]:
# Choose which dataset to run through the RNN
inputVar = inputVar_train   # or inputVar_test

T = inputVar.shape[1]  # 60 time steps

hidden_states = []
outputs = []

# Initial hidden state
hState = np.zeros((3, 1))

for t in range(T):

    # Current input vector (2 x 1)
    x_t = inputVar[:, t].reshape(2, 1)

    # Hidden state
    hState = tanh(
        np.dot(WxH, x_t)
        + np.dot(WhH, hState)
        + biasWV
    )

    # Output
    y_t = np.dot(WhY, hState) + biasWY

    hidden_states.append(hState)
    outputs.append(y_t)

# Stack results
hidden_states = np.hstack(hidden_states)   # (3, 60)
final_outputs = np.hstack(outputs)         # (2, 60)

print("Hidden States Shape:", hidden_states.shape)
print("Final Output Shape:", final_outputs.shape)

Final Output Shape: (2, 7)
